In [ ]:
from src.collectors.ccnews_extractor import (
    domain_allowed,
    url_is_article,
    load_processed,
    save_processed,
    load_month_paths,
    process_warc_stream,
    run_one_warc,
    perf_counter,
    Path,
    os,
    defaultdict
)

In [ ]:
BASE_URL = "https://data.commoncrawl.org/"

MONTHS = [
    "2025/01","2025/02","2025/03","2025/04","2025/05","2025/06",
    "2025/07","2025/08","2025/09","2025/10","2025/11","2025/12",
    "2026/01","2026/02","2026/03"
]

parent_dir = Path(os.curdir).resolve().parent
DATA_DIR = os.path.join(parent_dir, "data/raw/ccnews")
os.makedirs(DATA_DIR, exist_ok=True)

OUTPUT_FILE = os.path.join(DATA_DIR, "articles_filtered.jsonl")
PROCESSED_LOG = os.path.join(DATA_DIR, "processed_warcs_filtered.json")

MAX_HTML_SIZE = 1_000_000
WRITE_BATCH_SIZE = 200
MAX_ITEMS_PER_WARC = 100000
downsample_rate = None

ALLOWED_DOMAINS = {
    "reuters.com",
    "bbc.com",
    "bbc.co.uk",
    "ft.com",
    "bloomberg.com",
    "cnbc.com",
    "wsj.com",
    "nytimes.com",
    "economist.com",
    "theguardian.com",
    "washingtonpost.com",
    "apnews.com",
    "businessinsider.com",
    "marketwatch.com",
    "yahoo.com",
    "forbes.com",
    "cnn.com",
    "nbcnews.com",
    "abcnews.go.com",
    "aljazeera.com",

    # list of en domains
    'www.channelnewsasia.com',
    'www.thenationalnews.com',
    # 'www.swissinfo.ch',
    'www.straitstimes.com',
    'vietnamnews.vn',
    'abcnews.go.com',
    'www.cbsnews.com',
    'www.foxnews.com',
    'www.latimes.com',
    'globalnews.ca',
    'www.ctvnews.ca',
    'www.telegraph.co.uk',
    'www.independent.co.uk',
    'www.the-independent.com',
    'www.irishtimes.com',
    'www.scotsman.com',
    'www.yorkshirepost.co.uk',
    # 'timesofindia.indiatimes.com',
    # 'economictimes.indiatimes.com',
    'indianexpress.com',
    'www.ndtv.com',
    # 'www.business-standard.com',
    # 'www.livemint.com',
    # 'www.businesstoday.in',
    'gulfnews.com',
    'www.khaleejtimes.com',
    'allafrica.com',
    'www.timeslive.co.za',
    'www.businesslive.co.za',
    'thewest.com.au',
    'www.nzherald.co.nz',
    # 'www.nasdaq.com',
    # 'seekingalpha.com',
    'qz.com',
    # 'www.barchart.com',
    # 'www.pymnts.com',

    # ru domains
    'www.tass.ru',
    # 'ria.ru', # bad format of html
    'www.interfax.ru',
    'www.rbc.ru',
    'www.vedomosti.ru',
    'www.kommersant.ru',
    'www.lenta.ru',
    'www.nur.kz',
    'www.zakon.kz'
}

BAD_PATHS = [
    "/video/",
    "/videos/",
    "/live/",
    "/gallery/",
    "/podcast/",
    'sport',
    'crimea'
]

In [ ]:
t_total_start = perf_counter()
processed = load_processed(processed_log=PROCESSED_LOG)
month_paths = {m: load_month_paths(base_url=BASE_URL, month=m) for m in MONTHS}
total = defaultdict(float)

with open(OUTPUT_FILE, "a", encoding="utf-8") as out:
    while True:
        progress = False

        for m in MONTHS:
            if not month_paths[m]:
                continue

            warc_rel = month_paths[m].pop()

            if warc_rel in processed:
                continue

            url = BASE_URL + warc_rel
            print("Processing:", url)

            warc_stats = process_warc_stream(
                url,
                out,
                bad_paths=BAD_PATHS,
                max_html_size=MAX_HTML_SIZE,
                write_batch_size=WRITE_BATCH_SIZE,
                max_items_per_warc=MAX_ITEMS_PER_WARC,
                downsample_rate=downsample_rate,
                allowed_domains=ALLOWED_DOMAINS,
                allowed_langs=("en", "ru")
            )
            for key, value in warc_stats.items():
                total[key] += value

            print(
                "WARC stats | "
                f"records={int(warc_stats['records_seen'])} "
                f"responses={int(warc_stats['response_records'])} "
                f"url_ok={int(warc_stats['after_url_filters'])} "
                f"size_ok={int(warc_stats['after_size_filter'])} "
                f"extract_ok={int(warc_stats['extract_non_empty'])} "
                f"written={int(warc_stats['written'])} "
                f"limit_hit={int(warc_stats['stopped_by_limit'])} "
                f"lang_errors={int(warc_stats['lang_errors'])} "
                f"extract_s={warc_stats['extract_seconds']:.2f} "
                f"json_s={warc_stats['json_seconds']:.2f} "
                f"warc_s={warc_stats['warc_seconds']:.2f}"
            )

            processed.add(warc_rel)
            save_processed(processed, PROCESSED_LOG)
            progress = True

        if not progress:
            print("Finished all archives")
            break

total_seconds = perf_counter() - t_total_start
print(
    "Run summary | "
    f"records={int(total['records_seen'])} "
    f"responses={int(total['response_records'])} "
    f"url_ok={int(total['after_url_filters'])} "
    f"size_ok={int(total['after_size_filter'])} "
    f"extract_ok={int(total['extract_non_empty'])} "
    f"written={int(total['written'])} "
    f"json_errors={int(total['json_errors'])} "
    f"extract_s={total['extract_seconds']:.2f} "
    f"json_s={total['json_seconds']:.2f} "
    f"total_s={total_seconds:.2f}"
)


Processing: https://data.commoncrawl.org/crawl-data/CC-NEWS/2025/01/CC-NEWS-20250122093858-00433.warc.gz
WARC stats | records=54293 responses=27146 url_ok=658 size_ok=580 extract_ok=579 written=495 limit_hit=0 lang_errors=0 extract_s=18.39 json_s=0.01 warc_s=41.57
Processing: https://data.commoncrawl.org/crawl-data/CC-NEWS/2025/02/CC-NEWS-20250226165643-00908.warc.gz
WARC stats | records=46625 responses=23312 url_ok=725 size_ok=662 extract_ok=661 written=603 limit_hit=0 lang_errors=0 extract_s=24.90 json_s=0.01 warc_s=53.85
Processing: https://data.commoncrawl.org/crawl-data/CC-NEWS/2025/03/CC-NEWS-20250313181219-01141.warc.gz
WARC stats | records=49323 responses=24661 url_ok=1123 size_ok=1095 extract_ok=1095 written=1035 limit_hit=0 lang_errors=0 extract_s=46.87 json_s=0.02 warc_s=63.10
Processing: https://data.commoncrawl.org/crawl-data/CC-NEWS/2025/04/CC-NEWS-20250414053330-01622.warc.gz
WARC stats | records=43783 responses=21891 url_ok=672 size_ok=641 extract_ok=641 written=579 limit_hit=0 lang_errors=0 extract_s=25.03 json_s=0.01 warc_s=46.06

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed
import os
from collections import defaultdict
from time import perf_counter
import random
from src.collectors.ccnews_extractor import run_one_warc

cfg = {
    "bad_paths": BAD_PATHS,
    "max_html_size": MAX_HTML_SIZE,
    "write_batch_size": WRITE_BATCH_SIZE,
    "max_items_per_warc": MAX_ITEMS_PER_WARC,
    "downsample_rate": downsample_rate,
    "allowed_domains": ALLOWED_DOMAINS,
    "allowed_langs": ("en", "ru"),
    "output_root": DATA_DIR,
}

MAX_WORKERS = 4
t0 = perf_counter()

processed = load_processed(PROCESSED_LOG)
month_paths = {m: load_month_paths(BASE_URL, m) for m in MONTHS}

warc_rels = []
for m in MONTHS:
    warc_rels.extend(month_paths[m])
random.shuffle(warc_rels)

warc_rels = [w for w in warc_rels if w not in processed]

total = defaultdict(float)

with ProcessPoolExecutor(max_workers=MAX_WORKERS) as ex:
    future_map = {
        ex.submit(run_one_warc, BASE_URL + warc_rel, cfg): warc_rel
        for warc_rel in warc_rels
    }

    for fut in as_completed(future_map):
        warc_rel = future_map[fut]
        try:
            res = fut.result()
            stats = res["stats"]

            for k, v in stats.items():
                total[k] += v

            processed.add(warc_rel)
            save_processed(processed, PROCESSED_LOG)

            print(
                "WARC stats | "
                f"warc={warc_rel} "
                f"written={int(stats.get('written', 0))} "
                f"warc_s={stats.get('warc_seconds', 0):.2f} "
                f"out={res.get('output_path')}"
            )
        except Exception as e:
            print(f"FAILED {warc_rel}: {e}")

elapsed = perf_counter() - t0
print(
    "Run summary | "
    f"written={int(total.get('written', 0))} "
    f"records={int(total.get('records_seen', 0))} "
    f"extract_s={total.get('extract_seconds', 0):.2f} "
    f"total_s={elapsed:.2f}"
)

- перезапустить парсинг с новыми доменами
- сделать партицирование по месяцам в parquet файлы
- для очистки данных - посмотреть % статей по доменам с высоким уровнем повторов словосочетаний - фэйлы с парсингом html
